In [1]:
import os
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split

SEED = 42
TEST_SIZE = 0.20
OUT_DIR = "data/wikibio"
os.makedirs(OUT_DIR, exist_ok=True)

/home/shivraj-pg/miniconda3/envs/assignment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("potsawee/wiki_bio_gpt3_hallucination")
ds.save_to_disk(f"{OUT_DIR}/raw_hf")

Saving the dataset (1/1 shards): 100%|██████████| 238/238 [00:00<00:00, 18833.02 examples/s]


In [3]:
ds["evaluation"].features

{'gpt3_text': Value(dtype='string', id=None),
 'wiki_bio_text': Value(dtype='string', id=None),
 'gpt3_sentences': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
 'annotation': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
 'wiki_bio_test_idx': Value(dtype='int64', id=None),
 'gpt3_text_samples': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)}

In [4]:
df = ds["evaluation"].to_pandas()

In [5]:
df.shape

(238, 6)

In [6]:
df.head()

,gpt3_text,wiki_bio_text,gpt3_sentences,annotation,wiki_bio_test_idx,gpt3_text_samples
0,John Russell Reynolds (1820–1876) was an Engli...,"Sir John Russell Reynolds, 1st Baronet (22 May...",[John Russell Reynolds (1820–1876) was an Engl...,"[major_inaccurate, major_inaccurate, major_ina...",62464,[John Russell Reynolds (1 November 1829 – 11 ...
1,"Matthew Aylmer, 1st Baron Aylmer (1708–1794) w...","Admiral of the Fleet Matthew Aylmer, 1st Baron...","[Matthew Aylmer, 1st Baron Aylmer (1708–1794) ...","[minor_inaccurate, minor_inaccurate, minor_ina...",49661,"[""Matthew Aylmer, 1st Baron Aylmer (c. 1650–17..."
2,Rick Mahler (born Richard Alan Mahler on April...,"Richard Keith Mahler (August 5, 1953 in Austin...",[Rick Mahler (born Richard Alan Mahler on Apri...,"[minor_inaccurate, minor_inaccurate, accurate,...",20483,"[Rick Mahler (January 8, 1956 – May 25, 2005) ..."
3,James Blair (1732–1782) was an American lawyer...,"James Blair (September 26, 1786 - April 1, 183...",[James Blair (1732–1782) was an American lawye...,"[minor_inaccurate, major_inaccurate, major_ina...",71174,"[James Blair (April 2, 1755 – March 8, 1842) w..."
4,"Tim Finchem (born August 24, 1947) is an Ameri...","Timothy W. Finchem (born April 19, 1947) is th...","[Tim Finchem (born August 24, 1947) is an Amer...","[minor_inaccurate, accurate, major_inaccurate,...",39945,"[""Tim Finchem (born May 27, 1953) is an Americ..."


In [7]:
for c in df.columns:
    print(c)
    print("-" * 80)
    print(df[c][0])
    print("=" * 80)

gpt3_text
--------------------------------------------------------------------------------
John Russell Reynolds (1820–1876) was an English lawyer, judge, and author. He was born in London, the son of a barrister, and was educated at Eton College and Trinity College, Cambridge. He was called to the bar in 1845, and became a Queen's Counsel in 1859. He was appointed a judge of the Court of Common Pleas in 1867, and was knighted in 1871.

Reynolds was a prolific author, writing on a wide range of topics. He wrote several books on legal topics, including The Law of Libel and Slander (1863), The Law of Copyright (1865), and The Law of Patents for Inventions (1868). He also wrote on a variety of other topics, including history, biography, and literature. He was a frequent contributor to the Saturday Review, and wrote several books on Shakespeare, including The Mystery of William Shakespeare (1848) and The Authorship of Shakespeare (1875). He also wrote a biography of the poet John Keats (18

## Sentence-level table

**Change from the previous version:** the passage text and the 20 resamples are *not* repeated on every sentence row. They go into a separate passage-level table. (Repeating them made the CSV huge, and a CSV turns the list of resamples into a plain string, which would break Task 2.)

In [8]:
label_map = {"accurate": 0, "minor_inaccurate": 1, "major_inaccurate": 1}

sent_records, passage_records, problematic_rows = [], [], []

for bio_id, row in df.iterrows():
    sentences = row["gpt3_sentences"]
    annotations = row["annotation"]

    if len(sentences) != len(annotations):
        problematic_rows.append({"bio_id": bio_id,
                                 "num_sentences": len(sentences),
                                 "num_annotations": len(annotations)})
        continue

    passage_records.append({
        "bio_id": bio_id,
        "wiki_bio_test_idx": row["wiki_bio_test_idx"],
        "gpt3_text": row["gpt3_text"],
        "gpt3_text_samples": list(row["gpt3_text_samples"]),   # 20 resamples
    })

    for sentence_id, (sentence, annotation) in enumerate(zip(sentences, annotations)):
        annotation = str(annotation).strip().lower()
        if annotation not in label_map:
            raise ValueError(f"Unknown annotation at row {bio_id}: {annotation}")
        sent_records.append({
            "bio_id": bio_id,
            "sentence_id": sentence_id,
            "sentence": sentence,
            "annotation": annotation,
            "label": label_map[annotation],
        })

In [9]:
final_df = pd.DataFrame(sent_records)
passages_df = pd.DataFrame(passage_records)
problematic_df = pd.DataFrame(problematic_rows)

In [10]:
print("Original biographies:", len(df))
print("Successfully processed biographies:", len(passages_df))
print("Problematic biographies:", len(problematic_df))
print("Total processed sentences:", len(final_df))
print("Resamples per passage:", passages_df["gpt3_text_samples"].map(len).unique())

Original biographies: 238
Successfully processed biographies: 238
Problematic biographies: 0
Total processed sentences: 1908
Resamples per passage: [20]


In [11]:
final_df["label"].value_counts()

label
1    1392
0     516
Name: count, dtype: int64

In [12]:
print("Positive rate (= random-baseline AUC-PR):", round(final_df["label"].mean(), 4))

Positive rate (= random-baseline AUC-PR): 0.7296


### Splits

In [13]:
bio_ids = passages_df["bio_id"].values

train_bio_ids, test_bio_ids = train_test_split(bio_ids, test_size=TEST_SIZE, random_state=SEED)
train_bio_ids, test_bio_ids = set(train_bio_ids), set(test_bio_ids)
assert train_bio_ids.isdisjoint(test_bio_ids)
print("No biography overlap: Verified")

No biography overlap: Verified


In [14]:
final_df["split"] = np.where(final_df["bio_id"].isin(train_bio_ids), "train", "test")
passages_df["split"] = np.where(passages_df["bio_id"].isin(train_bio_ids), "train", "test")

In [15]:
print("Training biographies:", len(train_bio_ids), "| Testing biographies:", len(test_bio_ids))

Training biographies: 190 | Testing biographies: 48


In [16]:
final_df.groupby("split")["label"].agg(sentences="count", positive_rate="mean").round(3)

,sentences,positive_rate
split,,
test,383,0.762
train,1525,0.721


In [17]:
final_df["split"] = np.where(final_df["bio_id"].isin(train_bio_ids), "train", "test")
passages_df["split"] = np.where(passages_df["bio_id"].isin(train_bio_ids), "train", "test")

In [18]:
print("Training biographies:", len(train_bio_ids), "| Testing biographies:", len(test_bio_ids))

Training biographies: 190 | Testing biographies: 48


In [19]:
final_df.groupby("split")["label"].agg(sentences="count", positive_rate="mean").round(3)

,sentences,positive_rate
split,,
test,383,0.762
train,1525,0.721


In [20]:
final_df.to_csv(f"{OUT_DIR}/sentences.csv", index=False)
passages_df.to_parquet(f"{OUT_DIR}/passages.parquet", index=False)
problematic_df.to_csv(f"{OUT_DIR}/problematic_rows.csv", index=False)

DatasetDict({
    s: Dataset.from_pandas(final_df[final_df["split"] == s].reset_index(drop=True), preserve_index=False)
    for s in ["train", "test"]
}).save_to_disk(f"{OUT_DIR}/sentence_level_split")

Saving the dataset (1/1 shards): 100%|██████████| 383/383 [00:00<00:00, 164896.16 examples/s]
